# 07 — Intent Classification (`graph/intent.py`)
Two-stage classifier:
1. **LLM path** (Groq/OpenAI) — returns structured `IntentClassification` with `intent`, `data_products`, `confidence`, `reasoning`
2. **Keyword fallback** — pure regex, no API key needed, used in dev/CI

10 intent types: `full_diagnostic`, `data_quality`, `governance`, `incident_review`, `knowledge_lookup`, `metric_analysis`, `write_ticket`, `write_metadata`, `write_rule`, `unknown`


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'codebase', 'codebase', 'src'))
os.makedirs("logs", exist_ok=True)

## 1. Keyword Fallback (no API key needed)

In [ ]:
from graph.intent import _keyword_fallback, QueryIntent

tests = [
    "What is the GRR for retention?",
    "Why did retention drop last month?",
    "Who owns the bookings metric?",
    "Create a Jira ticket for the CAC anomaly",
    "What are open bugs for LTV data?",
    "Create a DQ rule for completeness",
    "Update the owner of bookings metric",
    "How does NRR differ from GRR?",
    "xyz abc 123",  # unknown
]

print(f"{'Query (truncated)':<48} {'Intent':<20} {'Products':<20} Conf")
print("-" * 100)
for q in tests:
    r = _keyword_fallback(q)
    disp = (q[:45] + "...") if len(q) > 48 else q
    print(f"{disp:<48} {r.intent.value:<20} {str(r.data_products):<20} {r.confidence:.2f}")

## 2. classify_intent_gpt (uses LLM if key set, else keyword fallback)

In [ ]:
from graph.intent import classify_intent_gpt

result = classify_intent_gpt("What is the gross retention rate for last month?")
print("intent      :", result.intent.value)
print("data_products:", result.data_products)
print("confidence  :", result.confidence)
print("reasoning   :", result.reasoning)

## 3. Convenience shims — used by supervisor_node

In [ ]:
from graph.intent import classify_intent, extract_products

# Returns just the intent string
intent = classify_intent("Why did bookings drop in Q3?")
print("intent:", intent)

# Returns just the products list
products = extract_products("Compare CAC payback vs LTV:CAC ratio")
print("products:", products)

## 4. IntentClassification Pydantic Schema

In [ ]:
from graph.intent import IntentClassification, QueryIntent
from pydantic import ValidationError

# Build manually (for testing)
ic = IntentClassification(
    intent=QueryIntent.METRIC_ANALYSIS,
    data_products=["retention", "cac"],
    confidence=0.92,
    reasoning="Query asks about metrics for two products",
)
print(ic.model_dump())

# Validation
try:
    bad = IntentClassification(intent="invalid_intent", data_products=[], confidence=2.0, reasoning="")
except ValidationError as e:
    print("\nValidation error (expected):", e.error_count(), "errors")

## 5. All 10 Intent Enum Values

In [ ]:
from graph.intent import QueryIntent
print("All intent values:")
for q in QueryIntent:
    print(f"  {q.value}")

## 6. INTENT_AGENT_MAP — What Gets Routed Where

In [ ]:
from graph.routing import INTENT_AGENT_MAP
print(f"{'Intent':<20} Agents")
print("-" * 60)
for intent, agents in INTENT_AGENT_MAP.items():
    print(f"{intent:<20} {agents}")

## 7. Product Keyword Mapping

In [ ]:
from graph.intent import _PRODUCT_KEYWORDS
print("Product keyword → product mapping:")
for kw, product in _PRODUCT_KEYWORDS.items():
    print(f"  '{kw}' → {product}")